# DisasterTweetLocator — full pipeline with results

Disaster tweet classification with **GloVe (6B.50d) + BiLSTM** and geolocation mapping with **spaCy NER + geopy + Folium**.

This notebook walks through every stage and shows the saved results:

1. Data and preprocessing
2. Train / validation / held-out test split, tokenizer and GloVe coverage
3. Model architecture
4. Evaluation of the saved BiLSTM model on the held-out test set
5. Training curves
6. **Table I**: BiLSTM vs BiGRU vs GRU vs LSTM (5 seeds each)
7. Ablation study (24 runs)
8. Location extraction and map plotting (the Table II tweets)

All numbers come from `train.py` (outputs saved in `outputs/`). Set `RETRAIN = True` below to retrain the main model from scratch.

In [1]:
import os, sys, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.insert(0, os.path.abspath(".."))   # repo root

import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import config
from preprocess import load_dataset, clean_text
from tokenizer import WordTokenizer
from embeddings import load_glove, build_embedding_matrix
from models import build_model, DISPLAY_NAMES
from train import split_data, evaluate

RETRAIN = False   # True = retrain the BiLSTM (seed 42) instead of loading outputs/bilstm_glove.keras
pd.set_option("display.max_colwidth", 110)
print("TensorFlow", tf.__version__)

I0000 00:00:1794465365.253702    3288 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1794465366.355063    3288 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.21.0


## 1. Data and preprocessing
Kaggle *NLP with Disaster Tweets*: 7,613 labelled tweets (`target` = 1 for a real disaster).

In [2]:
df = load_dataset(config.TRAIN_CSV)
print(df.shape)
print("Class balance:", df["target"].value_counts().rename({0: "normal", 1: "disaster"}).to_dict())
print("Tweets with an empty location field:", (df["location"] == "").mean().round(3))
df[["keyword", "location", "text", "target"]].head()

(7613, 6)
Class balance: {'normal': 4342, 'disaster': 3271}
Tweets with an empty location field: 0.333


,keyword,location,text,target
0,,,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,,,Forest fire near La Ronge Sask. Canada,1
2,,,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter i...,1
3,,,"13,000 people receive #wildfires evacuation orders in California",1
4,,,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1


Cleaning removes HTML tags, hyperlinks, square brackets, numbers, punctuation and non-ASCII symbols, then lower-cases the text (paper Sec. III-B).

In [3]:
examples = df[df["text"].str.contains("http|#|\\d", regex=True)].sample(5, random_state=0)
pd.DataFrame({"original": examples["text"], "cleaned": examples["clean_text"]})

,original,cleaned
1326,@minsuwoongs i completely understand because i just woke up like 15 minutes ago and im Burning,minsuwoongs i completely understand because i just woke up like minutes ago and im burning
1261,17 people displaced after 3-alarm fire tore through 2 apartment buildings on Second Street in Manchester. ...,people displaced after alarm fire tore through apartment buildings on second street in manchester mht
7279,Whirlwind Head Scissor on @alexhammerstone @kttape ktfounder #RemyMarcel #FroFroFroÛ_ https://t.co/B19z8V...,whirlwind head scissor on alexhammerstone kttape ktfounder remymarcel frofrofro
6084,Share Large sinkhole swallows entire pond in Lowndes County Georgia A largeÛ_ http://t.co/HvBJ30aj9s #YoNews,share large sinkhole swallows entire pond in lowndes county georgia a large yonews
7113,Dramatic Video Shows Plane Landing During Violent Storm http://t.co/XRgPdlSWfD,dramatic video shows plane landing during violent storm


## 2. Split, tokenizer and GloVe
20% of `train.csv` is held out as a **test set** that is never used for training or model selection. The remaining 80% is split into train and validation; validation is used only for early stopping.

In [4]:
train, val, test = split_data(df, config.MAIN_SEED)
print(f"train {len(train)} | validation {len(val)} | held-out test {len(test)}")

tok = WordTokenizer.load(config.OUTPUT_DIR / "tokenizer.json")
glove = load_glove(config.GLOVE_PATH, config.EMBEDDING_DIM)
emb, coverage = build_embedding_matrix(tok.word_index, glove, config.EMBEDDING_DIM, config.MAX_VOCAB)
print(f"vocabulary {emb.shape[0]} | embedding dim {emb.shape[1]} | words found in GloVe {coverage:.1%}")

enc = lambda d: tok.encode(d["clean_text"].tolist(), config.MAX_LEN)
X_te, y_te = enc(test), test["target"].to_numpy("int32")

train 5328 | validation 762 | held-out test 1523


vocabulary 10000 | embedding dim 50 | words found in GloVe 82.5%


## 3. Model architecture (paper Sec. III-D)
Embedding (GloVe, frozen) → BiLSTM → GlobalMaxPooling1D → Dense(ReLU) → Dropout → Dense(sigmoid). Adam (lr = 0.001), binary cross-entropy.

In [5]:
if RETRAIN:
    tf.keras.utils.set_random_seed(config.MAIN_SEED)
    model = build_model("bilstm", emb)
    X_tr, y_tr = enc(train), train["target"].to_numpy("int32")
    X_val, y_val = enc(val), val["target"].to_numpy("int32")
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=config.EARLY_STOP_PATIENCE, restore_best_weights=True)
    history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=config.EPOCHS,
                        batch_size=config.BATCH_SIZE, callbacks=[es], verbose=2).history
else:
    model = tf.keras.models.load_model(config.OUTPUT_DIR / "bilstm_glove.keras")
    history = json.loads((config.OUTPUT_DIR / "history_bilstm.json").read_text())
model.summary()

Model: "bilstm_glove"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ glove_embedding (Embedding)     │ (None, 50, 50)         │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 50, 256)        │       183,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pool                 │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_relu (Dense)              │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,099,429 (4.19 MB)

 Trainable params: 199,809 (780.50 KB)

 Non-trainable params: 500,000 (1.91 MB)

 Optimizer params: 399,620 (1.52 MB)

## 4. Evaluation on the held-out test set (BiLSTM + GloVe, seed 42)

In [6]:
m = evaluate(model, X_te, y_te)
print(f"Accuracy  {m['accuracy']:.4f}\nPrecision {m['precision']:.4f}\nRecall    {m['recall']:.4f}\nF1 score  {m['f1']:.4f}\n")
pred = (model.predict(X_te, verbose=0).ravel() >= 0.5).astype(int)
print(classification_report(y_te, pred, target_names=["normal", "disaster"], digits=4))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=["normal", "disaster"]).plot(cmap="Blues", colorbar=False)
plt.title("BiLSTM + GloVe — held-out test"); plt.show()

Accuracy  0.8168
Precision 0.8641
Recall    0.6804
F1 score  0.7613



              precision    recall  f1-score   support

      normal     0.7927    0.9194    0.8514       869
    disaster     0.8641    0.6804    0.7613       654

    accuracy                         0.8168      1523
   macro avg     0.8284    0.7999    0.8063      1523
weighted avg     0.8233    0.8168    0.8127      1523



## 5. Training curves (Fig. 4 and Fig. 5)
Early stopping restores the weights from the epoch with the lowest validation loss.

In [7]:
ep = range(1, len(history["loss"]) + 1)
best = int(np.argmin(history["val_loss"])) + 1
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, history["val_loss"], "o-", color="#c0392b"); ax[0].axvline(best, ls="--", c="grey", label=f"best epoch {best}")
ax[0].set_title("Validation loss"); ax[0].legend()
ax[1].plot(ep, history["loss"], "o-", color="#2471a3"); ax[1].set_title("Training loss")
for a in ax: a.set_xlabel("Epoch"); a.set_ylabel("Binary cross-entropy"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"validation loss: {history['val_loss'][0]:.4f} (epoch 1) -> {min(history['val_loss']):.4f} (epoch {best})")

validation loss: 0.4600 (epoch 1) -> 0.4354 (epoch 5)


## 6. Table I — model comparison
Each model was trained with 5 seeds (42, 1, 2, 3, 4) and scored on the same held-out test set. Mean ± standard deviation.

In [8]:
print((config.OUTPUT_DIR / "results_table.md").read_text())
runs = pd.DataFrame([json.loads(l) for l in (config.OUTPUT_DIR / "runs.jsonl").read_text().splitlines()])
per_seed = runs.assign(model=runs["model"].map(DISPLAY_NAMES),
                       test_accuracy=runs["test"].map(lambda t: round(t["accuracy"], 4)),
                       test_f1=runs["test"].map(lambda t: round(t["f1"], 4)))
per_seed.pivot(index="model", columns="seed", values="test_accuracy")

Held-out test set (20% of train.csv, n=1523). Mean ± sd over seeds.

| Model          | Accuracy (%)   |   Precision |   Recall |   F1 Score |   Seeds |
|:---------------|:---------------|------------:|---------:|-----------:|--------:|
| BiLSTM + GloVe | 81.62 ± 0.81   |      0.8158 |   0.7413 |     0.7757 |       5 |
| BiGRU + GloVe  | 81.90 ± 0.99   |      0.8277 |   0.7327 |     0.7767 |       5 |
| GRU + GloVe    | 80.83 ± 1.58   |      0.8143 |   0.726  |     0.7641 |       5 |
| LSTM + GloVe   | 80.38 ± 1.39   |      0.8174 |   0.7067 |     0.7551 |       5 |



seed,1,2,3,4,42
model,,,,,
BiGRU + GloVe,0.8293,0.8175,0.8030,0.8214,0.8240
BiLSTM + GloVe,0.8201,0.8247,0.8162,0.8030,0.8168
GRU + GloVe,0.8240,0.8214,0.7912,0.7919,0.8129
LSTM + GloVe,0.8194,0.8155,0.7919,0.7879,0.8043


In [9]:
order = [DISPLAY_NAMES[v] for v in ["bilstm", "bigru", "gru", "lstm"]]
data = [per_seed.loc[per_seed["model"] == n, "test_accuracy"] * 100 for n in order]
plt.figure(figsize=(7, 4)); plt.boxplot(data, tick_labels=order); plt.ylabel("Test accuracy (%)")
plt.title("Held-out accuracy across 5 seeds"); plt.grid(axis="y", alpha=.3); plt.show()

## 7. Ablation study
24 runs of the BiLSTM + GloVe model from the original notebook configuration (`experiments/ablation/ablation.py`), varying text preprocessing, model size, dropout, sequence length, vocabulary size, batch size, learning rate and embedding fine-tuning. Selection used a validation split; accuracy below is on a separate held-out test split.

In [10]:
abl = pd.DataFrame([json.loads(l) for l in open("../experiments/ablation/ablation_results.jsonl")])
abl["setting"] = abl["cfg"].map(lambda c: ", ".join(f"{k}={v}" for k, v in c.items()) or "original notebook config")
abl["val_acc"] = abl["val"].map(lambda v: round(v["acc"], 4))
abl["test_acc"] = abl["test"].map(lambda v: round(v["acc"], 4))
print(f"{len(abl)} runs | test accuracy range {abl.test_acc.min():.4f} - {abl.test_acc.max():.4f} | mean {abl.test_acc.mean():.4f}")
base = abl[abl["setting"] == "original notebook config"]["test_acc"]
print(f"original config over {len(base)} seeds: {base.mean():.4f} ± {base.std():.4f}")
abl[["setting", "seed", "val_acc", "test_acc"]].sort_values("test_acc", ascending=False).reset_index(drop=True)

24 runs | test accuracy range 0.8135 - 0.8326 | mean 0.8218
original config over 5 seeds: 0.8218 ± 0.0038


,setting,seed,val_acc,test_acc
0,dedupe=True,42,0.7966,0.8326
1,lr=0.0005,42,0.7992,0.8293
2,units=64,42,0.8005,0.8280
3,spatial=0.2,42,0.8136,0.8260
4,finetune_epochs=3,42,0.7940,0.8260
5,"spatial=0.2, text=raw_kw, dedupe=True, max_words=20000, batch=32, epochs=25, patience=4, finetune_epochs=3",42,0.8123,0.8253
6,max_words=20000,42,0.8045,0.8247
7,original notebook config,2,0.8031,0.8247
8,original notebook config,3,0.8018,0.8247
9,batch=32,42,0.8110,0.8240


## 8. Location extraction and map plotting (Table II)
Tweets classified as disasters are passed to spaCy `en_core_web_sm`; GPE entities are geocoded with geopy (Nominatim) and plotted with Folium. If Nominatim can't be reached, a small offline cache (`data/geocode_cache.json`) is used for these demo places.

In [11]:
from geolocate import extract_locations, plot_tweets
from predict_and_map import TABLE_II_TWEETS

probs = model.predict(tok.encode([clean_text(t) for t in TABLE_II_TWEETS], config.MAX_LEN), verbose=0).ravel()
rows, records = [], []
for t, p in zip(TABLE_II_TWEETS, probs):
    locs = extract_locations(t) if p >= 0.5 else []
    rows.append({"tweet": t, "P(disaster)": round(float(p), 3), "label": "disaster" if p >= 0.5 else "normal",
                 "locations (spaCy GPE)": ", ".join(locs) or "—"})
    if locs: records.append({"text": t, "probability": float(p), "locations": locs})
pd.DataFrame(rows)

,tweet,P(disaster),label,locations (spaCy GPE)
0,"Snowstorm in Denver! Whiteout conditions, highways closed, and locals hunkering down",0.856,disaster,Denver
1,"Tornado warning in Oklahoma City! Sirens blaring, skies darkening, and residents seeking shelter.",0.999,disaster,Oklahoma City
2,"Flooding devastates New Orleans, Louisiana, as levees breach and neighborhoods are submerged. Residents ev...",0.996,disaster,"New Orleans, Louisiana"
3,"Severe flooding in Malawi displaces thousands, devastates crops. Relief efforts are underway. #MalawiFlood...",0.993,disaster,—
4,"Just had the best pizza of my life, this place is fire!",0.090,normal,—


In [12]:
fmap = plot_tweets(records, config.OUTPUT_DIR / "disaster_map.html")
print("Interactive map saved to outputs/disaster_map.html (open it in a browser)")
fmap

Interactive map saved to outputs/disaster_map.html (open it in a browser)


**Observation.** The three US tweets are geolocated. The Malawi tweet is classified as a disaster, but spaCy's small model doesn't tag *Malawi* as a GPE, so it can't be plotted. This is the same limitation reported in Table II of the paper, and a larger NER model (e.g. `en_core_web_trf`) is a natural improvement.